# End-Term Project: Weather Data Analysis and Visualization
**Dataset:**  (Daily meteorological data for Kazakhstan regions in 2024)  


---
## Project Objectives
1. **Data Cleaning & Preprocessing:** Check and clean missing values, duplicate entries, data types, and inspect for outliers.
2. **Task 1:** Calculate and visualize the average temperature for each month.
3. **Task 2:** Identify the region with the most precipitation and compare total precipitation across all regions.
4. **Task 3:** Visualize daily temperature trends and moving averages for a selected region (Almaty).
5. **Task 4:** Analyze correlation between temperature, wind speed, and precipitation using a heatmap.
6. **Task 5:** Determine the exact day with the most stable temperature for each region using day-to-day absolute change.

## Step 0: Setup, Data Cleaning & Preprocessing (3 Points)
In this initial step, we import the necessary libraries, load , inspect summary statistics, handle any missing or duplicate records, inspect for outliers, and convert the date column into datetime format.

In [ ]:
# Import pandas for data manipulation and tabular analysis
import pandas as pd

# Import numpy for numerical computations
import numpy as np

# Import matplotlib.pyplot for plotting charts
import matplotlib.pyplot as plt

# Import seaborn for statistical data visualization
import seaborn as sns

# Set plot style
sns.set_theme(style="whitegrid")

# Load the weather dataset from CSV into a pandas DataFrame
df = pd.read_csv("weather.csv")

print("--- Step 0.1: Initial Inspection of the Weather Dataset ---")
print("Dataset Shape (Rows, Columns):", df.shape)
print()
print("First 5 rows of the dataset:")
print(df.head())
print()
print("Dataset Information:")
df.info()


In [ ]:
print("--- Step 0.2: Handling Missing Values, Duplicates, and Dates ---")

# 1. Check for missing values
missing_counts = df.isnull().sum()
print("Missing values per column:")
print(missing_counts)
print()

if missing_counts.sum() > 0:
    df = df.dropna()
    print(f"Removed missing values.")
else:
    print("No missing values found in the dataset!")

# 2. Check for duplicate rows
duplicate_count = df.duplicated().sum()
print("Duplicate rows found:", duplicate_count)
if duplicate_count > 0:
    df = df.drop_duplicates()
    print(f"Removed {duplicate_count} duplicate rows.")
else:
    print("No duplicate rows found!")
print()

# 3. Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Extract month number (1-12) and month name (Jan-Dec)
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")

print("Date conversion completed. New columns added:")
print(df[["date", "region", "month", "month_name"]].head())


In [ ]:
print("--- Step 0.3: Outlier Check via Descriptive Statistics and Boxplots ---")
print(df[["temperature_c", "precipitation_mm", "wind_speed_kmh"]].describe())
print()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Boxplot for Temperature
sns.boxplot(y=df["temperature_c"], ax=axes[0], color="skyblue")
axes[0].set_title("Boxplot of Temperature (°C)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Temperature (°C)")

# Boxplot for Precipitation
sns.boxplot(y=df["precipitation_mm"], ax=axes[1], color="lightgreen")
axes[1].set_title("Boxplot of Precipitation (mm)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Precipitation (mm)")

# Boxplot for Wind Speed
sns.boxplot(y=df["wind_speed_kmh"], ax=axes[2], color="salmon")
axes[2].set_title("Boxplot of Wind Speed (km/h)", fontsize=12, fontweight="bold")
axes[2].set_ylabel("Wind Speed (km/h)")

plt.tight_layout()
plt.show()

print("Outlier Verification:")
print("- Temperature values range from around -21°C to +39°C, consistent with continental climate ranges.")
print("- Precipitation is non-negative (min 0.0 mm) and follows an expected right-skewed precipitation distribution.")
print("- Wind speed values are all realistic (max ~35 km/h). No erroneous data points require removal.")


## Task 1: Average Temperature for Each Month
We group the weather data by month and compute the average temperature in Celsius across all regions.

In [ ]:
# Group by month and calculate the average temperature
monthly_temp = df.groupby(["month", "month_name"])["temperature_c"].mean().reset_index()
monthly_temp = monthly_temp.sort_values("month")

print("--- Task 1: Average Monthly Temperature ---")
for _, row in monthly_temp.iterrows():
    print(f"{row['month_name']:<5} (Month {int(row['month']):>2}): {row['temperature_c']:>6.2f} °C")
print()

plt.figure(figsize=(10, 5))
plt.bar(
    monthly_temp["month_name"],
    monthly_temp["temperature_c"],
    color="cornflowerblue",
    edgecolor="black",
    alpha=0.85,
    label="Average Temperature"
)
plt.plot(
    monthly_temp["month_name"],
    monthly_temp["temperature_c"],
    color="crimson",
    marker="o",
    linewidth=2.5,
    label="Seasonal Curve"
)
plt.axhline(0, color="gray", linestyle="--", linewidth=1.2, label="Freezing Line (0°C)")
plt.title("Average Monthly Temperature in 2024", fontsize=14, fontweight="bold")
plt.xlabel("Month", fontsize=12)
plt.ylabel("Average Temperature (°C)", fontsize=12)
plt.legend(loc="upper left")
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


## Task 2: Region with the Most Precipitation
We group data by  and calculate the total precipitation sum across the analyzed year.

In [ ]:
# Group by region and sum precipitation
region_precip = df.groupby("region")["precipitation_mm"].sum().reset_index()
region_precip.columns = ["region", "total_precipitation_mm"]
region_precip_sorted = region_precip.sort_values(by="total_precipitation_mm", ascending=False)

print("--- Task 2: Total Precipitation by Region ---")
for _, row in region_precip_sorted.iterrows():
    print(f"{row['region']:<12}: {row['total_precipitation_mm']:>7.2f} mm")
print()

top_region = region_precip_sorted.iloc[0]["region"]
top_val = region_precip_sorted.iloc[0]["total_precipitation_mm"]
print(f"RESULT: The region with the most precipitation is {top_region} ({top_val:.2f} mm).")
print()

plt.figure(figsize=(9, 5))
palette = ["#2b5c8f", "#4682b4", "#5c9edc", "#87ceeb", "#b0e0e6"]
bars = plt.bar(
    region_precip_sorted["region"],
    region_precip_sorted["total_precipitation_mm"],
    color=palette,
    edgecolor="black"
)
for b in bars:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width()/2.0, h + 5, f"{h:.1f} mm", ha="center", va="bottom", fontweight="bold")

plt.title("Total Precipitation by Region (2024)", fontsize=14, fontweight="bold")
plt.xlabel("Region", fontsize=12)
plt.ylabel("Total Precipitation (mm)", fontsize=12)
plt.ylim(0, region_precip_sorted["total_precipitation_mm"].max() * 1.15)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


## Task 3: Temperature Trends for a Selected Region (Almaty)
We filter the dataset for Almaty and visualize daily temperatures throughout 2024.

In [ ]:
selected_region = "Almaty"
almaty_df = df[df["region"] == selected_region].sort_values(by="date").copy()

print(f"--- Task 3: Temperature Trends for {selected_region} ---")
print(f"Days analyzed: {len(almaty_df)}")
print(f"Minimum temperature: {almaty_df['temperature_c'].min():.1f} °C")
print(f"Maximum temperature: {almaty_df['temperature_c'].max():.1f} °C")
print(f"Mean annual temperature: {almaty_df['temperature_c'].mean():.1f} °C")
print()

plt.figure(figsize=(13, 5))
plt.plot(almaty_df["date"], almaty_df["temperature_c"], color="#d95f02", linewidth=1.2, alpha=0.8, label="Daily Temperature")
almaty_df["temp_7day_rolling"] = almaty_df["temperature_c"].rolling(window=7, center=True).mean()
plt.plot(almaty_df["date"], almaty_df["temp_7day_rolling"], color="navy", linewidth=2.0, label="7-Day Moving Average")
plt.axhline(0, color="black", linestyle="--", linewidth=1.0, alpha=0.7, label="0°C Freezing Line")
plt.title(f"Daily Temperature Trend in {selected_region} (2024)", fontsize=14, fontweight="bold")
plt.xlabel("Date", fontsize=12)
plt.ylabel("Temperature (°C)", fontsize=12)
plt.legend(loc="upper left")
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


## Task 4: Correlation Between Temperature, Wind Speed, and Precipitation
We calculate the correlation matrix between the 3 continuous numeric variables and visualize it with a Seaborn heatmap.

In [ ]:
numeric_columns = ["temperature_c", "precipitation_mm", "wind_speed_kmh"]
corr_matrix = df[numeric_columns].corr()

print("--- Task 4: Correlation Matrix ---")
print(corr_matrix.round(3))
print()

plt.figure(figsize=(7, 5))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    vmin=-1.0, vmax=1.0,
    square=True,
    linewidths=1.0,
    cbar_kws={"shrink": 0.8}
)
plt.title("Correlation Matrix: Temperature, Wind Speed & Precipitation", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Interpretation:")
print("The correlation values are close to 0, indicating little to no linear relationship between temperature, precipitation, and wind speed in this dataset.")


## Task 5: Day with the Most Stable Temperature in Each Region
**Definition:** Since each region has one observation per day, we define the most stable day as the date on which the day-to-day absolute temperature difference $|T_t - T_{t-1}|$ is the smallest.

In [ ]:
df_sorted = df.sort_values(by=["region", "date"]).copy()
df_sorted["prev_day_temp"] = df_sorted.groupby("region")["temperature_c"].shift(1)
df_sorted["daily_temp_diff"] = (df_sorted["temperature_c"] - df_sorted["prev_day_temp"]).abs()

df_valid = df_sorted.dropna(subset=["daily_temp_diff"])
unique_regions = df_valid["region"].unique()

stable_results = []
for reg in unique_regions:
    reg_df = df_valid[df_valid["region"] == reg]
    min_idx = reg_df["daily_temp_diff"].idxmin()
    best_row = reg_df.loc[min_idx]
    stable_results.append({
        "Region": reg,
        "Date": best_row["date"].strftime("%Y-%m-%d"),
        "Today_Temp_C": best_row["temperature_c"],
        "Yesterday_Temp_C": best_row["prev_day_temp"],
        "Abs_Diff_C": round(best_row["daily_temp_diff"], 3)
    })

stable_df = pd.DataFrame(stable_results)
print("=== DAY WITH THE MOST STABLE TEMPERATURE IN EACH REGION ===")
print(stable_df.to_string(index=False))


## Summary & Project Conclusion
- **Cleaning:** 1,830 rows with 0 missing and 0 duplicate values. Date column successfully parsed.
- **Seasonality:** Summer months (June-August) show highest average temperatures, while winter months (December-February) drop below freezing.
- **Precipitation:** Regional totals identified the highest rainfall region clearly.
- **Stability:** Identified the exact date for each region with the minimal daily temperature fluctuation.